In [4]:
import os
import langchain
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain.document_loaders import WebBaseLoader
from concurrent.futures import ThreadPoolExecutor
import requests
from bs4 import BeautifulSoup

def fetch_paper_by_title(title):
    try:
        # Clean title for search
        query = title.replace(" ", "+")
        
        # URLs for searching on arXiv and Google Scholar
        search_urls = [
            f"https://arxiv.org/search/?query={query}&searchtype=all&source=header",
            f"https://scholar.google.com/scholar?q={query}"
        ]
        
        def fetch_url(url):
            try:
                # Get search page
                response = requests.get(url, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")
                
                # For arXiv: find first paper link
                if "arxiv.org" in url:
                    link = soup.find("a", class_="arxiv-url")
                    if link and "abs" in link["href"]:
                        return link["href"]
                # For Google Scholar: find first paper link
                elif "scholar.google.com" in url:
                    link = soup.find("h3", class_="gs_rt")
                    if link and link.find("a"):
                        return link.find("a")["href"]
                return None
            except:
                return None

        # Search in parallel
        with ThreadPoolExecutor(max_workers=2) as executor:
            results = list(executor.map(fetch_url, search_urls))
        
        # Get the first valid URL
        paper_url = next((url for url in results if url), None)
        
        if not paper_url:
            st.error("No paper found for the given title.")
            return None, None
        
        # Load paper content
        loader = WebBaseLoader(paper_url)
        documents = loader.load()
        chunks = text_splitter.split_documents(documents)
        
        st.session_state.documents = documents
        st.session_state.chunks = chunks
        
        return documents, chunks
    except Exception as e:
        st.error(f"Error fetching paper: {str(e)}")
        return None, None

In [26]:
# Library
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.schema import HumanMessage
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

In [6]:
API = os.environ.get('GROQ')

In [11]:
def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    return loader.load()


In [12]:
def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)


In [27]:
def create_vectorstore(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    return vectorstore

In [29]:
config = '''You are an expert researcher and educator. Given the following academic paper, provide a comprehensive, yet concise summary covering all of the following:

1. 🧠 **Core Objective**: What is the main goal or research question of this paper?
2. 🆕 **Novel Contributions**: What are the key innovations or new ideas proposed by the authors?
3. 📊 **Important Results or Charts**: Describe the most important results, visualizations, or tables. Mention what they represent and why they matter.
4. 🛠️ **Methodology (Simple)**: Briefly explain how the study was conducted or the methods used — in simple, clear language.
5. 🔍 **Key Findings**: What did the authors discover? Summarize the conclusions.
6. 💡 **Why It Matters**: Explain the importance or impact of this research in its field.
7. 🧩 **Limitations or Open Questions**: Mention any limitations or future work suggested.

Present the output in **numbered bullet points**, in a format easy for non-experts to understand.

Start by saying:  
**“Here’s a deep breakdown of the paper in simple terms:”**
'''

In [ ]:
def get_qa_chain(vectorstore):
    llm = ChatGroq(api_key = API, temperature=0.2, model_name="llama3-70b-8192")
    test_prompt = [HumanMessage(content=config)]
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever()
    )
    response = llm.invoke(test_prompt)
    print(response.content)


In [46]:
get_top_pdf_links_from_scholar('Attention is all you need.')

[]

PDF Link: None


In [15]:
import requests
from bs4 import BeautifulSoup

def extract_clean_text_from_html(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    # Remove script and style elements
    for tag in soup(["script", "style", "noscript", "footer", "header", "form", "nav"]):
        tag.decompose()

    # Extract text from <article> if available, else from <main>, else body
    main_content = soup.find("article") or soup.find("main") or soup.body
    if not main_content:
        return "No main content found."

    # Join and clean text
    text = main_content.get_text(separator="\n", strip=True)
    return text



ok = extract_clean_text_from_html('https://journals.aai.org/bib/article/25/1/bbad467/7512647')

In [ ]:
import requests

url = "https://research-io.onrender.com/search"
payload = {
    "query": "Attention is all you need",
    "max_results": 3
}

resp = requests.post(url, json=payload)
print(resp.json())


<Response [200]>
200
{'results': [{'title': 'Transformers: Learning with Purely Attention Based Networks', 'pdf_link': 'https://engineering.purdue.edu/kak/pdf-kak/Transformers.pdf'}, {'title': 'How Much Attention is "All You Need"?', 'pdf_link': 'https://web.stanford.edu/class/cs224n/final-reports/256987290.pdf'}]}


In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

def selenium_fetch_pdfs_with_titles(query, max_results=3):
    start_time = time.time()

    options = Options()
    options.add_argument("--headless=new")  # Use new headless mode
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-images")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36")

    driver = webdriver.Chrome(options=options)

    formatted_query = f'{query} (site:arxiv.org OR site:researchgate.net OR site:*.edu OR site:*.ac.in OR site:*.ac.uk OR site:ieeexplore.ieee.org OR site:springer.com) filetype:pdf'
    search_url = f"https://www.google.com/search?q={formatted_query.replace(' ', '+')}"
    driver.get(search_url)

    time.sleep(1)  # Wait for JS to load

    results = []
    try:
        result_blocks = driver.find_elements(By.CSS_SELECTOR, "div.b8lM7")

        for block in result_blocks:
            try:
                title_el = block.find_element(By.CSS_SELECTOR, "h3.LC20lb.MBeuO.DKV0Md")
                title = title_el.text.strip()

                link_el = block.find_element(By.TAG_NAME, "a")
                href = link_el.get_attribute("href")

                if href and href.endswith(".pdf"):
                    results.append({
                        "title": title,
                        "pdf_link": href
                    })

                if len(results) >= max_results:
                    break

            except Exception:
                continue

    except Exception as e:
        print("[Selenium Error]", e)

    driver.quit()
    print(f"[Selenium] Time taken: {round(time.time() - start_time, 2)} seconds")
    return results

# Run it
if __name__ == "__main__":
    query = "Open agent architecture"
    links = selenium_fetch_pdfs_with_titles(query)
    print("Results:\n")
    for item in links:
        print(f"Title: {item['title']}\nPDF: {item['pdf_link']}\n")


[Selenium] Time taken: 6.59 seconds
Title: Attention Is All You Need In Speech Separation
PDF: https://ieeexplore.ieee.org/iel7/9413349/9413350/09413901.pdf

Title: A Little Bit Attention Is All You Need for Person Re- ...
PDF: https://ieeexplore.ieee.org/iel7/10160211/10160212/10160304.pdf

